## ℹ️ Using Pre-Trained Model

**Current Status**: The dashboard is configured to use **pre-trained LSTM models** from `keras-trained model/` folder.

**What This Means**:
- ✅ Dashboard loads pre-trained models automatically
- ✅ No need to run this notebook for dashboard functionality
- ✅ Run this notebook only if you want to:
  - Retrain models with new data
  - Experiment with different architectures
  - Validate model performance

**Dashboard Status**: 
- Model Status: ✅ Using Pre-Trained Models
- Models Location: `keras-trained model/` folder
- Ready for: Demand forecasting, anomaly detection, real-time predictions

For quick start guide, see: [QUICK_START_PRETRAINED.md](../QUICK_START_PRETRAINED.md)

In [ ]:
# ========================================
# DEMAND FORECASTING MODEL - KAGGLE
# LSTM Neural Network for Smart Meter Data
# ========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib
from pathlib import Path

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("✓ All libraries imported successfully")

In [ ]:
# ========================================
# 1. LOAD ALL DATASET FILES FROM ROOT
# ========================================

print("\n" + "="*70)
print("LOADING REAL DATASETS FROM FOLDERS")
print("="*70)

# Use local datasets path
DATASETS_PATH = Path('../datasets')

print(f"\nDataset location: {DATASETS_PATH.absolute()}")

# Find all CSV files
csv_files = sorted(list(DATASETS_PATH.glob('*.csv')))
print(f"\nFound {len(csv_files)} CSV files:")
for i, file in enumerate(csv_files, 1):
    print(f"  {i}. {file.name}")

# Load all CSV files with column standardization
print("\nLoading all files...")
dfs = []
for file in csv_files:
    try:
        df = pd.read_csv(file)
        
        # Standardize column names based on actual dataset format
        if 'x_Timestamp' in df.columns:
            df = df.rename(columns={'x_Timestamp': 'timestamp'})
        elif 'Date' in df.columns:
            df = df.rename(columns={'Date': 'timestamp'})
        
        if 't_kWh' in df.columns:
            df = df.rename(columns={'t_kWh': 'consumption_kWh'})
        
        # Standardize meter column
        if 'meter' in df.columns:
            df = df.rename(columns={'meter': 'meter_id'})
        
        dfs.append(df)
        print(f"  ✓ {file.name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"  ✗ {file.name}: Error - {str(e)}")

print(f"\n✓ Total files loaded: {len(dfs)}")

In [ ]:
# Combine ALL dataframes
df_combined = pd.concat(dfs, ignore_index=True, sort=False)
print(f"\n✓ Successfully loaded all files!")
print(f"Combined dataset shape: {df_combined.shape}")
print(f"Total records: {len(df_combined):,}")
print(f"\nColumns in combined dataset:")
print(df_combined.columns.tolist())

# Drop rows with missing values in key columns
print(f"\nBefore cleanup:")
print(f"  Missing values: {df_combined.isnull().sum().sum()}")
print(f"  Records: {len(df_combined):,}")

df_combined = df_combined.dropna()

print(f"After cleanup:")
print(f"  Missing values: {df_combined.isnull().sum().sum()}")
print(f"  Records: {len(df_combined):,}")
print(f"\nFinal columns: {df_combined.columns.tolist()}")

In [ ]:
# ========================================
# 2. DATA PREPROCESSING
# ========================================

print("\n" + "="*70)
print("DATA PREPROCESSING WITH REAL DATASET")
print("="*70)

# Use the actual column names from the dataset
timestamp_col = 'timestamp'
consumption_col = 'consumption_kWh'

print(f"\nTimestamp column: {timestamp_col}")
print(f"Consumption column: {consumption_col}")

# Convert timestamp to datetime
df_combined[timestamp_col] = pd.to_datetime(df_combined[timestamp_col])

# Sort by timestamp
df_combined = df_combined.sort_values(timestamp_col).reset_index(drop=True)

print(f"\nData range: {df_combined[timestamp_col].min()} to {df_combined[timestamp_col].max()}")
print(f"Total duration: {(df_combined[timestamp_col].max() - df_combined[timestamp_col].min()).days} days")

# Remove outliers (Interquartile Range method)
Q1 = df_combined[consumption_col].quantile(0.25)
Q3 = df_combined[consumption_col].quantile(0.75)
IQR = Q3 - Q1

print(f"\nOutlier detection (IQR method):")
print(f"  Q1: {Q1:.4f}, Q3: {Q3:.4f}, IQR: {IQR:.4f}")

df_combined = df_combined[(df_combined[consumption_col] >= Q1 - 1.5*IQR) & 
                          (df_combined[consumption_col] <= Q3 + 1.5*IQR)]

print(f"  Records after outlier removal: {len(df_combined):,}")

# Fill any remaining gaps in time series
df_combined = df_combined.sort_values(timestamp_col).reset_index(drop=True)

print(f"\n✓ Preprocessing complete!")

In [ ]:
# ========================================
# 3. FEATURE ENGINEERING
# ========================================

print("\n" + "="*70)
print("FEATURE ENGINEERING")
print("="*70)

# Aggregate by date (daily consumption)
df_agg = df_combined.groupby(df_combined[timestamp_col].dt.date)[consumption_col].sum().reset_index()
df_agg.columns = ['date', 'consumption']
df_agg['date'] = pd.to_datetime(df_agg['date'])
df_agg = df_agg.sort_values('date').reset_index(drop=True)

print(f"Aggregated data shape: {df_agg.shape}")
print(f"Date range: {df_agg['date'].min()} to {df_agg['date'].max()}")

# Create time-based features
df_agg['day_of_week'] = df_agg['date'].dt.dayofweek
df_agg['month'] = df_agg['date'].dt.month
df_agg['quarter'] = df_agg['date'].dt.quarter
df_agg['day_of_month'] = df_agg['date'].dt.day
df_agg['week_of_year'] = df_agg['date'].dt.isocalendar().week

# Create lag features (past 7 days)
for lag in range(1, 8):
    df_agg[f'lag_{lag}'] = df_agg['consumption'].shift(lag)

# Create rolling average features
for window in [7, 14, 30]:
    df_agg[f'rolling_mean_{window}'] = df_agg['consumption'].rolling(window=window).mean()

# Drop rows with NaN values
df_agg = df_agg.dropna().reset_index(drop=True)

print(f"Final feature set shape: {df_agg.shape}")


In [ ]:
# ========================================
# 4. PREPARE DATA FOR MODELING
# ========================================

print("\n" + "="*70)
print("PREPARING DATA FOR MODELING")
print("="*70)

feature_cols = [col for col in df_agg.columns if col not in ['date', 'consumption']]
X = df_agg[feature_cols]
y = df_agg['consumption']

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")

# Scale features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Scale target
scaler_y = MinMaxScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


In [ ]:
# ========================================
# 4b. RESHAPE DATA FOR LSTM
# ========================================

print("\n" + "="*70)
print("RESHAPING DATA FOR LSTM (3D FORMAT)")
print("="*70)

# Reshape from (n_samples, n_features) to (n_samples, 1, n_features)
X_train_reshaped = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_reshaped = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print(f"Original shape: {X_train.shape}")
print(f"Reshaped for LSTM: {X_train_reshaped.shape}")
print(f"  Samples: {X_train_reshaped.shape[0]}")
print(f"  Timesteps: {X_train_reshaped.shape[1]}")
print(f"  Features: {X_train_reshaped.shape[2]}")

In [ ]:
# ========================================
# 5. BUILD LSTM MODEL
# ========================================

print("\n" + "="*70)
print("BUILDING LSTM MODEL")
print("="*70)

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])),
    Dropout(0.2),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print(f"Model parameters: {model.count_params():,}")
print("\nModel architecture:")
model.summary()

In [ ]:
# ========================================
# 6. TRAIN MODEL (100 EPOCHS)
# ========================================

print("\n" + "="*70)
print("TRAINING MODEL WITH 100 EPOCHS")
print("="*70)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_reshaped, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("\n✓ Model training complete")

In [ ]:
# ========================================
# 7. EVALUATE MODEL
# ========================================

print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

y_train_pred = model.predict(X_train_reshaped, verbose=0)
y_test_pred = model.predict(X_test_reshaped, verbose=0)

# Metrics on test set (scaled)
train_mse = mean_squared_error(y_train, y_train_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\nTraining Set Metrics (Scaled):")
print(f"  MSE: {train_mse:.4f}")
print(f"  MAE: {train_mae:.4f}")
print(f"  R²: {train_r2:.4f}")

print(f"\nTest Set Metrics (Scaled):")
print(f"  MSE: {test_mse:.4f}")
print(f"  MAE: {test_mae:.4f}")
print(f"  R²: {test_r2:.4f}")

In [ ]:
# Inverse transform to original scale
y_train_original = scaler_y.inverse_transform(y_train.reshape(-1, 1)).flatten()
y_test_original = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_train_pred_original = scaler_y.inverse_transform(y_train_pred).flatten()
y_test_pred_original = scaler_y.inverse_transform(y_test_pred).flatten()

# Metrics on original scale
train_rmse_original = np.sqrt(mean_squared_error(y_train_original, y_train_pred_original))
test_rmse_original = np.sqrt(mean_squared_error(y_test_original, y_test_pred_original))

print(f"\nOriginal Scale (kWh):")
print(f"  Train RMSE: {train_rmse_original:.2f} kWh")
print(f"  Test RMSE: {test_rmse_original:.2f} kWh")

In [ ]:
# ========================================
# 8. VISUALIZATIONS
# ========================================

print("\n" + "="*70)
print("GENERATING VISUALIZATIONS")
print("="*70)

# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Model Loss Over Epochs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('Model MAE Over Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Predictions vs Actual
fig, ax = plt.subplots(figsize=(14, 6))

test_indices = range(len(y_test_original))
ax.plot(test_indices, y_test_original, 'o-', label='Actual', linewidth=2, markersize=4, alpha=0.7)
ax.plot(test_indices, y_test_pred_original, 's-', label='Predicted', linewidth=2, markersize=4, alpha=0.7)

ax.set_xlabel('Test Sample Index')
ax.set_ylabel('Consumption (kWh)')
ax.set_title(f'Test Set: Predictions vs Actual (RMSE: {test_rmse_original:.2f} kWh)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# 9. SAVE TRAINED MODEL FILES
# ========================================

print("\n" + "="*70)
print("SAVING TRAINED MODEL & ARTIFACTS")
print("="*70)

# Create outputs directory in project root
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir.absolute()}")

# Save trained model (KERAS format)
model_path = output_dir / 'demand_forecasting_lstm.keras'
model.save(str(model_path))
print(f"✓ Trained model saved: {model_path}")
print(f"  Size: {model_path.stat().st_size / (1024*1024):.2f} MB")

# Save scalers
scaler_X_path = output_dir / 'scaler_X.pkl'
scaler_y_path = output_dir / 'scaler_y.pkl'
joblib.dump(scaler_X, str(scaler_X_path))
joblib.dump(scaler_y, str(scaler_y_path))
print(f"✓ Scalers saved")

# Save feature columns
features_path = output_dir / 'feature_columns.pkl'
joblib.dump(feature_cols, str(features_path))
print(f"✓ Feature columns saved: {len(feature_cols)} features")

# Save model metadata
metadata = {
    'model_type': 'LSTM',
    'feature_columns': feature_cols,
    'timestamp_column': timestamp_col,
    'consumption_column': consumption_col,
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'total_samples': len(df_agg),
    'input_shape': str(X_train_reshaped.shape),
    'train_r2': float(train_r2),
    'test_r2': float(test_r2),
    'train_rmse': float(train_rmse_original),
    'test_rmse': float(test_rmse_original),
    'epochs_trained': len(history.history['loss']),
    'creation_date': str(pd.Timestamp.now()),
    'dataset_info': {
        'total_raw_records': len(df_combined),
        'aggregated_records': len(df_agg),
        'date_range': f"{df_agg['date'].min()} to {df_agg['date'].max()}",
        'duration_days': (df_agg['date'].max() - df_agg['date'].min()).days
    }
}

metadata_path = output_dir / 'model_metadata.pkl'
joblib.dump(metadata, str(metadata_path))
print(f"✓ Metadata saved")

print(f"\n✓ All model files saved to: {output_dir.absolute()}")

In [ ]:
# ========================================
# VERIFY ALL FILES WERE SAVED
# ========================================

print("\n" + "="*70)
print("VERIFYING SAVED FILES IN OUTPUT DIRECTORY")
print("="*70)

saved_files = list(output_dir.glob('*.keras')) + list(output_dir.glob('*.pkl'))
print(f"\n✓ Total files in output directory: {len(saved_files)}")

total_size = 0
for file in sorted(saved_files):
    file_size = file.stat().st_size / (1024 * 1024)  # Convert to MB
    total_size += file_size
    print(f"  ✓ {file.name} ({file_size:.2f} MB)")

print(f"\nTotal size: {total_size:.2f} MB")

if len(saved_files) == 5:
    print("\n✓✓✓ ALL 5 FILES SAVED SUCCESSFULLY ✓✓✓")
    print("✓ Dashboard will auto-load these models!")
else:
    print(f"\n⚠️ Expected 5 files, found {len(saved_files)}")

In [ ]:
# ========================================
# 10. FINAL SUMMARY REPORT
# ========================================

print("\n" + "="*70)
print("DEMAND FORECASTING MODEL - TRAINING COMPLETE")
print("="*70)

summary = f"""
📊 USING REAL DATASET FROM /datasets FOLDER
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DATASET INFORMATION:
  • Data source: Local datasets folder
  • Total raw records: {len(df_combined):,}
  • Aggregated records: {len(df_agg):,}
  • Date range: {df_agg['date'].min()} to {df_agg['date'].max()}
  • Duration: {(df_agg['date'].max() - df_agg['date'].min()).days} days

FEATURES CREATED:
  • Total features: {len(feature_cols)}
  • Time-based: day_of_week, month, quarter, day_of_month, week_of_year
  • Lag features: lag_1 to lag_7
  • Rolling averages: 7, 14, 30 day windows

DATA PREPARATION:
  • Training samples: {len(X_train):,}
  • Test samples: {len(X_test):,}
  • Input shape (3D): {X_train_reshaped.shape}
  • Timesteps: {X_train_reshaped.shape[1]}
  • Features per timestep: {X_train_reshaped.shape[2]}

MODEL ARCHITECTURE:
  • Type: LSTM Neural Network
  • Layers: 3 LSTM (128 → 64 → 32 units) + Dense layers
  • Dropout: 0.2 (regularization)
  • Total parameters: {model.count_params():,}
  • Optimizer: Adam (lr=0.001)
  • Loss function: Mean Squared Error

TRAINING CONFIGURATION:
  • Epochs: {len(history.history['loss'])}/100
  • Early stopping: Yes (patience=10)
  • Batch size: 32
  • Validation split: 20%

PERFORMANCE METRICS:
  • Train R² Score: {train_r2:.4f}
  • Test R² Score: {test_r2:.4f}
  ✓ Test RMSE: {test_rmse_original:.2f} kWh
  ✓ Train RMSE: {train_rmse_original:.2f} kWh
  • Test MAE: {test_mae:.4f} (scaled)

SAVED ARTIFACTS (in /outputs):
  ✓ demand_forecasting_lstm.keras (trained model)
  ✓ scaler_X.pkl (feature scaler)
  ✓ scaler_y.pkl (target scaler)
  ✓ feature_columns.pkl (feature list)
  ✓ model_metadata.pkl (model info & metrics)

NEXT STEP:
  → Start dashboard: streamlit run app.py
  → Dashboard will auto-load these trained models!
  → Forecasting page will use REAL trained LSTM
  → Status: ✅ Using Pre-Trained Models

STATUS: ✓ TRAINING COMPLETE - Ready for deployment!
"""

print(summary)